In [1]:
# ===================================
# XSS DEFENSE TOOL - SETUP
# ===================================
# This installs required libraries

import sys
!{sys.executable} -m pip install requests beautifulsoup4 --quiet

print("✅ Setup complete!")
print("All libraries installed.")

✅ Setup complete!
All libraries installed.


In [21]:
import requests
# ===================================
# AUTO-DETECT FRAMEWORK
# ===================================
def detect_framework(url: str) -> str:
    """
    Automatically detects if website uses Django, Flask, or other framework
    """
    try:
        response = requests.get(url, timeout=10)
        
        # Clue 1: Check cookies
        cookies = response.cookies
        if 'csrftoken' in cookies or 'sessionid' in cookies:
            return "Django"
        if 'session' in cookies:
            return "Flask"
        
        # Clue 2: Check headers
        headers = response.headers
        server = headers.get('Server', '').lower()
        
        if 'werkzeug' in server:
            return "Flask"
        
        # Django clue: WSGIServer + X-Frame-Options DENY
        if 'wsgiserver' in server and headers.get('X-Frame-Options') == 'DENY':
            return "Django"
        
        # Clue 3: Check HTML content
        html = response.text.lower()
        if 'django' in html[:5000]:
            return "Django"
        if 'flask' in html[:5000]:
            return "Flask"
        
        return "Generic Website"
        
    except Exception as e:
        return f"Unknown (Error: {e})"

# Test it!
print("✅ Updated framework detector ready!")

# Test on your Django app
test_url = "http://127.0.0.1:8000/safe/"
framework = detect_framework(test_url)
print(f"\nURL: {test_url}")
print(f"Framework detected: {framework}")

✅ Updated framework detector ready!

URL: http://127.0.0.1:8000/safe/
Framework detected: Django


In [23]:
# ===================================
# DJANGO XSS SCANNER
# ===================================

import os
import re

def scan_django_source_code(project_path: str):
    """
    Scans Django project source code for XSS vulnerabilities
    
    What it looks for:
    1. mark_safe() usage
    2. |safe filter in templates  
    3. Markup() usage
    4. render_to_string without escape
    """
    
    print("\n" + "="*70)
    print("🔍 DJANGO SOURCE CODE SCANNER")
    print("="*70)
    print(f"Scanning: {project_path}")
    print("="*70)
    
    vulnerabilities = []
    
    # Dangerous patterns to look for
    patterns = patterns = {
    'mark_safe': {
        'regex': r'mark_safe\s*\(',
        'severity': 'HIGH',
        'description': 'mark_safe() disables auto-escaping - XSS risk!',
        'recommendation': 'Remove mark_safe() and let Django auto-escape',
    },
    'safe_filter': {
        'regex': r'\|\s*safe',
        'severity': 'HIGH', 
        'description': '|safe filter disables escaping - XSS risk!',
        'recommendation': 'Remove |safe filter',
    },
    'safestring': {
        'regex': r'SafeString\s*\(',
        'severity': 'HIGH',
        'description': 'SafeString() marks content as safe - XSS risk!',
        'recommendation': 'Avoid SafeString, use auto-escaping',
    },
    'format_html': {
        'regex': r'format_html\s*\(',
        'severity': 'MEDIUM',
        'description': 'format_html with user input can be risky',
        'recommendation': 'Ensure all variables are properly escaped',
    },
    'json_script_unsafe': {
        'regex': r'json_script\s*\(',
        'severity': 'LOW',
        'description': 'json_script - verify no XSS in JSON data',
        'recommendation': 'Ensure JSON data is sanitized',
    },
    'autoescape_off': {
        'regex': r'{%\s*autoescape\s+off\s*%}',
        'severity': 'HIGH',
        'description': 'autoescape off disables protection - XSS risk!',
        'recommendation': 'Remove autoescape off, use default on',
    },
}
    
    # Walk through all Python and HTML files
    for root, dirs, files in os.walk(project_path):
        for file in files:
            if file.endswith(('.py', '.html')):
                file_path = os.path.join(root, file)
                
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        lines = content.split('\n')
                        
                        # Check each pattern
                        for pattern_name, pattern_info in patterns.items():
                            matches = re.finditer(pattern_info['regex'], content)
                            
                            for match in matches:
                                # Find line number
                                line_num = content[:match.start()].count('\n') + 1
                                line_content = lines[line_num - 1].strip()
                                
                                vulnerabilities.append({
                                    'file': file_path,
                                    'line': line_num,
                                    'code': line_content,
                                    'pattern': pattern_name,
                                    'severity': pattern_info['severity'],
                                    'description': pattern_info['description'],
                                    'recommendation': pattern_info['recommendation'],
                                })
                except:
                    pass  # Skip files that can't be read
    
    # Display results
    if not vulnerabilities:
        print("\n✅ No XSS vulnerabilities found in source code!")
        return vulnerabilities
    
    print(f"\n⚠️  Found {len(vulnerabilities)} potential XSS vulnerability(ies):\n")
    
    for i, vuln in enumerate(vulnerabilities, 1):
        print(f"\n[{i}] {vuln['severity']} SEVERITY")
        print(f"    File: {vuln['file']}")
        print(f"    Line: {vuln['line']}")
        print(f"    Code: {vuln['code'][:80]}...")
        print(f"    Issue: {vuln['description']}")
        print(f"    Fix: {vuln['recommendation']}")
        print("-" * 70)
    
    return vulnerabilities

# Test it!
print("✅ Django scanner ready!")

✅ Django scanner ready!


In [24]:
# ===================================
# TEST DJANGO SCANNER
# ===================================

# Path to your Django project
django_project_path = r"E:\BKPP\Desktop\xss_test_django"

# Run the scanner
vulnerabilities = scan_django_source_code(django_project_path)

# Summary
print("\n" + "="*70)
print("SCAN SUMMARY")
print("="*70)
print(f"Total vulnerabilities found: {len(vulnerabilities)}")

if vulnerabilities:
    high = sum(1 for v in vulnerabilities if v['severity'] == 'HIGH')
    medium = sum(1 for v in vulnerabilities if v['severity'] == 'MEDIUM')
    low = sum(1 for v in vulnerabilities if v['severity'] == 'LOW')
    
    print(f"  HIGH:   {high}")
    print(f"  MEDIUM: {medium}")
    print(f"  LOW:    {low}")
    print("\n⚠️  ACTION REQUIRED: Fix these vulnerabilities!")
else:
    print("✅ No vulnerabilities detected!")

print("="*70)


🔍 DJANGO SOURCE CODE SCANNER
Scanning: E:\BKPP\Desktop\xss_test_django

⚠️  Found 1 potential XSS vulnerability(ies):


[1] HIGH SEVERITY
    File: E:\BKPP\Desktop\xss_test_django\vulnerable_app\templates\vulnerable.html
    Line: 5
    Code: <p>{{ user_input|safe }}</p>...
    Issue: |safe filter disables escaping - XSS risk!
    Fix: Remove |safe filter
----------------------------------------------------------------------

SCAN SUMMARY
Total vulnerabilities found: 1
  HIGH:   1
  MEDIUM: 0
  LOW:    0

⚠️  ACTION REQUIRED: Fix these vulnerabilities!


In [25]:
# ===================================
# DJANGO URL XSS TESTER
# ===================================

import requests

def test_django_url_xss(url: str, param: str = 'name'):
    """
    Tests a Django URL for XSS vulnerabilities
    
    Sends XSS payloads and checks if they're executed
    """
    
    print("\n" + "="*70)
    print("🌐 DJANGO URL XSS TESTER")
    print("="*70)
    print(f"Testing: {url}")
    print(f"Parameter: {param}")
    print("="*70)
    
    # XSS test payloads
    payloads = [
        "<script>alert('XSS')</script>",
        "<img src=x onerror=alert('XSS')>",
        "{{7*7}}",  # Template injection test
    ]
    
    vulnerabilities = []
    
    for payload in payloads:
        try:
            # Send payload
            response = requests.get(url, params={param: payload}, timeout=10)
            
            # Check if payload is reflected without escaping
            if payload in response.text:
                vulnerabilities.append({
                    'payload': payload,
                    'evidence': 'Payload reflected without encoding',
                    'severity': 'HIGH',
                })
                print(f"\n⚠️  VULNERABLE!")
                print(f"   Payload: {payload}")
                print(f"   Evidence: Payload appears in HTML without encoding")
            
            # Check for HTML encoding (safe)
            elif '&lt;' in response.text or '&gt;' in response.text:
                print(f"\n✅ SAFE")
                print(f"   Payload: {payload}")
                print(f"   Evidence: Properly HTML encoded")
        
        except Exception as e:
            print(f"\n❌ Error testing {payload}: {e}")
    
    print("\n" + "="*70)
    if vulnerabilities:
        print(f"⚠️  Found {len(vulnerabilities)} XSS vulnerability(ies)")
        print("\nRECOMMENDATION:")
        print("  • Remove |safe filter from templates")
        print("  • Remove mark_safe() from views")
        print("  • Let Django's auto-escaping protect you")
    else:
        print("✅ No XSS vulnerabilities found in URL testing")
    print("="*70)
    
    return vulnerabilities

print("✅ URL tester ready!")

✅ URL tester ready!


In [26]:
# ===================================
# TEST DJANGO URLs
# ===================================

print("\n" + "🔴"*35)
print("TESTING VULNERABLE DJANGO URL")
print("🔴"*35)

vulnerable_results = test_django_url_xss(
    url="http://127.0.0.1:8000/vulnerable/",
    param="name"
)

print("\n\n" + "🟢"*35)
print("TESTING SAFE DJANGO URL")
print("🟢"*35)

safe_results = test_django_url_xss(
    url="http://127.0.0.1:8000/safe/",
    param="name"
)

# Final comparison
print("\n" + "="*70)
print("COMPARISON")
print("="*70)
print(f"Vulnerable endpoint: {len(vulnerable_results)} XSS issue(s)")
print(f"Safe endpoint:       {len(safe_results)} XSS issue(s)")
print("="*70)


🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴
TESTING VULNERABLE DJANGO URL
🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴

🌐 DJANGO URL XSS TESTER
Testing: http://127.0.0.1:8000/vulnerable/
Parameter: name

⚠️  VULNERABLE!
   Payload: <script>alert('XSS')</script>
   Evidence: Payload appears in HTML without encoding

⚠️  VULNERABLE!
   Payload: <img src=x onerror=alert('XSS')>
   Evidence: Payload appears in HTML without encoding

⚠️  VULNERABLE!
   Payload: {{7*7}}
   Evidence: Payload appears in HTML without encoding

⚠️  Found 3 XSS vulnerability(ies)

RECOMMENDATION:
  • Remove |safe filter from templates
  • Remove mark_safe() from views
  • Let Django's auto-escaping protect you


🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
TESTING SAFE DJANGO URL
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢

🌐 DJANGO URL XSS TESTER
Testing: http://127.0.0.1:8000/safe/
Parameter: name

✅ SAFE
   Payload: <script>alert('XSS')</script>
   Evidence: Properly HTML encoded

✅ SAFE
   Payload: <img src=x onerror=alert('XSS')>
   Evidenc

In [27]:
# ===================================
# FLASK XSS SCANNER
# ===================================

def scan_flask_source_code(project_path: str):
    """
    Scans Flask project source code for XSS vulnerabilities
    """
    
    print("\n" + "="*70)
    print("🔍 FLASK SOURCE CODE SCANNER")
    print("="*70)
    print(f"Scanning: {project_path}")
    print("="*70)
    
    vulnerabilities = []
    
    # Dangerous patterns for Flask
    patterns = {
        'Markup': {
            'regex': r'Markup\s*\(',
            'severity': 'HIGH',
            'description': 'Markup() disables auto-escaping - XSS risk!',
            'recommendation': 'Remove Markup() and let Jinja2 auto-escape',
        },
        'safe_filter': {
            'regex': r'\|\s*safe',
            'severity': 'HIGH',
            'description': '|safe filter disables escaping - XSS risk!',
            'recommendation': 'Remove |safe filter from templates',
        },
        'render_template_string': {
            'regex': r'render_template_string\s*\(',
            'severity': 'HIGH',
            'description': 'render_template_string with user input - XSS/SSTI risk!',
            'recommendation': 'Use render_template() with files instead',
        },
        'autoescape_false': {
            'regex': r'autoescape\s*=\s*False',
            'severity': 'HIGH',
            'description': 'autoescape=False disables protection - XSS risk!',
            'recommendation': 'Remove autoescape=False, use default True',
        },
    }
    
    # Scan files
    for root, dirs, files in os.walk(project_path):
        for file in files:
            if file.endswith(('.py', '.html')):
                file_path = os.path.join(root, file)
                
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        lines = content.split('\n')
                        
                        for pattern_name, pattern_info in patterns.items():
                            matches = re.finditer(pattern_info['regex'], content)
                            
                            for match in matches:
                                line_num = content[:match.start()].count('\n') + 1
                                line_content = lines[line_num - 1].strip()
                                
                                vulnerabilities.append({
                                    'file': file_path,
                                    'line': line_num,
                                    'code': line_content,
                                    'pattern': pattern_name,
                                    'severity': pattern_info['severity'],
                                    'description': pattern_info['description'],
                                    'recommendation': pattern_info['recommendation'],
                                })
                except:
                    pass
    
    # Display results
    if not vulnerabilities:
        print("\n✅ No XSS vulnerabilities found in source code!")
        return vulnerabilities
    
    print(f"\n⚠️  Found {len(vulnerabilities)} potential XSS vulnerability(ies):\n")
    
    for i, vuln in enumerate(vulnerabilities, 1):
        print(f"\n[{i}] {vuln['severity']} SEVERITY")
        print(f"    File: {vuln['file']}")
        print(f"    Line: {vuln['line']}")
        print(f"    Code: {vuln['code'][:80]}...")
        print(f"    Issue: {vuln['description']}")
        print(f"    Fix: {vuln['recommendation']}")
        print("-" * 70)
    
    return vulnerabilities

print("✅ Flask scanner ready!")

✅ Flask scanner ready!


In [28]:
# ===================================
# TEST FLASK SCANNER
# ===================================

# Path to your Flask project
flask_project_path = r"E:\BKPP\Desktop\xss_test_flask"

# Run the scanner
flask_vulnerabilities = scan_flask_source_code(flask_project_path)

# Summary
print("\n" + "="*70)
print("FLASK SCAN SUMMARY")
print("="*70)
print(f"Total vulnerabilities found: {len(flask_vulnerabilities)}")

if flask_vulnerabilities:
    high = sum(1 for v in flask_vulnerabilities if v['severity'] == 'HIGH')
    medium = sum(1 for v in flask_vulnerabilities if v['severity'] == 'MEDIUM')
    low = sum(1 for v in flask_vulnerabilities if v['severity'] == 'LOW')
    
    print(f"  HIGH:   {high}")
    print(f"  MEDIUM: {medium}")
    print(f"  LOW:    {low}")
    print("\n⚠️  ACTION REQUIRED: Fix these vulnerabilities!")
else:
    print("✅ No vulnerabilities detected!")

print("="*70)


🔍 FLASK SOURCE CODE SCANNER
Scanning: E:\BKPP\Desktop\xss_test_flask

⚠️  Found 2 potential XSS vulnerability(ies):


[1] HIGH SEVERITY
    File: E:\BKPP\Desktop\xss_test_flask\app.py
    Line: 6
    Code: # VULNERABLE: Using Markup (disables auto-escaping)...
    Issue: Markup() disables auto-escaping - XSS risk!
    Fix: Remove Markup() and let Jinja2 auto-escape
----------------------------------------------------------------------

[2] HIGH SEVERITY
    File: E:\BKPP\Desktop\xss_test_flask\app.py
    Line: 10
    Code: html = Markup(f"<h1>Hello {user_input}</h1>")  # DANGEROUS!...
    Issue: Markup() disables auto-escaping - XSS risk!
    Fix: Remove Markup() and let Jinja2 auto-escape
----------------------------------------------------------------------

FLASK SCAN SUMMARY
Total vulnerabilities found: 2
  HIGH:   2
  MEDIUM: 0
  LOW:    0

⚠️  ACTION REQUIRED: Fix these vulnerabilities!


In [29]:
# ===================================
# TEST FLASK URLs
# ===================================

print("\n" + "🔴"*35)
print("TESTING VULNERABLE FLASK URL")
print("🔴"*35)

flask_vulnerable_results = test_django_url_xss(
    url="http://127.0.0.1:5000/vulnerable/",
    param="name"
)

print("\n\n" + "🟢"*35)
print("TESTING SAFE FLASK URL")
print("🟢"*35)

flask_safe_results = test_django_url_xss(
    url="http://127.0.0.1:5000/safe/",
    param="name"
)

# Final comparison
print("\n" + "="*70)
print("FLASK COMPARISON")
print("="*70)
print(f"Vulnerable endpoint: {len(flask_vulnerable_results)} XSS issue(s)")
print(f"Safe endpoint:       {len(flask_safe_results)} XSS issue(s)")
print("="*70)


🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴
TESTING VULNERABLE FLASK URL
🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴

🌐 DJANGO URL XSS TESTER
Testing: http://127.0.0.1:5000/vulnerable/
Parameter: name

⚠️  VULNERABLE!
   Payload: <script>alert('XSS')</script>
   Evidence: Payload appears in HTML without encoding

⚠️  VULNERABLE!
   Payload: <img src=x onerror=alert('XSS')>
   Evidence: Payload appears in HTML without encoding

⚠️  VULNERABLE!
   Payload: {{7*7}}
   Evidence: Payload appears in HTML without encoding

⚠️  Found 3 XSS vulnerability(ies)

RECOMMENDATION:
  • Remove |safe filter from templates
  • Remove mark_safe() from views
  • Let Django's auto-escaping protect you


🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
TESTING SAFE FLASK URL
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢

🌐 DJANGO URL XSS TESTER
Testing: http://127.0.0.1:5000/safe/
Parameter: name

⚠️  VULNERABLE!
   Payload: <script>alert('XSS')</script>
   Evidence: Payload appears in HTML without encoding

⚠️  VULNERABLE!
   Payload: <img src